In [ ]:
import teehr
import pandas as pd
from teehr.evaluation.spark_session_utils import create_spark_session

from teehr import DeterministicMetrics as dm
from teehr import Signatures as s
from teehr import RowLevelCalculatedFields as rcf
from teehr import TimeseriesAwareCalculatedFields as tcf
from teehr import Bootstrappers as bs

from teehr.models.filters import TableFilter

from pyspark.sql import functions as F

from pyspark.sql import DataFrame

import copy
import time

teehr.__version__

# Usage
- Configure the AWS 'default' profile for a user that has warehouse read/write permissions
- Adjust the `reference_time` timestamps in `filters` to your requested time window (current implementation supports quarters, e.g. Jan01-Mar31)

# Process
Starting with the joined timeseries table:
- Filter down to the configurations and time periods of interest
- Add row level calulated fields such as `forecast_leadtime_bin` and `quarter` (one new column), and time series aware calculated fields such as threshold exceedence (one column per threshold).  This requires the joined timeseries uniqueness fields be used. Total number of rows is unchanged.
- Stack (un-pivot) the data such that there is a threshold column that contains the threshold exceeded.  This increses the total number of rows of data, but allows our normal grouping/aggregates to be used.
- Aggregate based on the uniqueness fields plus row level calulated fields and threshold, to determine the min/median/max of each primary and secondary value within each `forecast_leadtime_bin` for each forecast.  This reduces the number of rows and adds a column per aggregation.
- Stack (un-pivot) the data again such that the min/mean/max columns are called primary_value and secondary_value and there is a `window_agg` column to identify which the value is.
- Lastly, we can group by location_id, configuration_name, etc. plus `forecast_leadtime_bin` and `window_agg` and calculate metrics with and without bootstrapping.

In [ ]:
import os

# Alternate executor pod template targeting the ON-DEMAND `nb-r5-4xlarge-teehr`
# node group instead of the spot `spark-r5-4xlarge-spot` pool, for tuning runs
# where we want clean measurements without spot-interruption noise. Same
# instance type (r5.4xlarge) so executor sizing math stays comparable to prior
# spot-based runs. Different taint on this node group (hub.jupyter.org/dedicated
# =user vs teehr-hub/dedicated=worker), so it needs its own tolerations.
ONDEMAND_POD_TEMPLATE_PATH = os.path.expanduser("~/executor-pod-template-ondemand.yaml")

with open(ONDEMAND_POD_TEMPLATE_PATH, "w") as f:
    f.write("""apiVersion: v1
kind: Pod
spec:
  terminationGracePeriodSeconds: 60
  securityContext:
    runAsUser: 1000
    runAsGroup: 1000
    fsGroup: 1000
  containers:
  - name: spark-kubernetes-executor
    securityContext:
      runAsUser: 1000
      runAsGroup: 1000
      allowPrivilegeEscalation: false
    lifecycle:
      preStop:
        exec:
          command: ["/bin/sh", "-c", "sleep 30"]
    volumeMounts:
    - name: data-nfs
      mountPath: /data
  volumes:
  - name: data-nfs
    persistentVolumeClaim:
      claimName: data-nfs
  tolerations:
  - effect: "NoSchedule"
    key: "hub.jupyter.org/dedicated"
    operator: "Equal"
    value: "user"
  - effect: "NoSchedule"
    key: "hub.jupyter.org_dedicated"
    operator: "Equal"
    value: "user"
  nodeSelector:
    teehr-hub/nodegroup-name: nb-r5-4xlarge
""")

print(f"Wrote alternate pod template to {ONDEMAND_POD_TEMPLATE_PATH}")


In [ ]:
# spark = create_spark_session(
#     start_spark_cluster=True,
#     executor_instances=64,
#     executor_memory="16g",
#     executor_cores=2,
#     aws_profile="default",
#     pod_template_path=ONDEMAND_POD_TEMPLATE_PATH,
#     update_configs={
#         "spark.sql.shuffle.partitions": 1024,
#         "spark.sql.adaptive.coalescePartitions.enabled": "false",
#         "spark.kubernetes.executor.annotation.cluster-autoscaler.kubernetes.io/safe-to-evict": "false",
#         "spark.executorEnv.TEEHR_BOOTSTRAP_ENGINE": "vectorized",
#         "spark.executor.memoryOverhead": "4g",
#     }
# )

spark = create_spark_session()

In [ ]:
spark.sql("DROP TABLE IF EXISTS nwmd_metrics_by_location_test PURGE")

In [ ]:
start = time.perf_counter()

In [ ]:
ev = teehr.RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

In [ ]:
joined_cols = ev.table("fcst_joined_timeseries").to_sdf().columns
non_unique_fields = ['primary_value','secondary_value','created_at','updated_at', "value_time"]
uniquenes_fields = [c for c in joined_cols if c not in non_unique_fields]
# uniquenes_fields

In [ ]:
ids = ev.locations.filter("id like 'usgs-%'").to_sdf().select("id")
sample = ids.sample(False, 0.5, seed=456).limit(10).collect()
location_ids = [r.id for r in sample]
print(len(location_ids))

# spark.sql("""
# USE iceberg.teehr
# """)
# rows = spark.sql("""
# SELECT distinct primary_location_id FROM fcst_joined_timeseries
# """).collect()
# location_ids = [r.primary_location_id for r in rows]
# print(len(location_ids))

In [ ]:
configurations = [
    {
        "configurations": ["nwm30_medium_range"],
        "forecast_lead_time_bin_hours": 24,
        "start_reference_time": "2025-10-01T00:00",
        "end_reference_time": "2026-10-01T00:00"
    },
    {
        "configurations": ["nwm30_short_range"],
        "forecast_lead_time_bin_hours": 6,
        "start_reference_time": "2025-10-01T00:00",
        "end_reference_time": "2026-10-01T00:00"
    }
]

In [ ]:
filters = [
    TableFilter(
        column="configuration_name",
        operator="=",
        value="nwm30_short_range"
    ),
    TableFilter(
        column="reference_time",
        operator=">=",
        value="2025-10-01T00:00",
    ),
    TableFilter(
        column="reference_time",
        operator="<",
        value="2026-10-01T00:00",
    ),
    TableFilter(
        column="primary_location_id",
        operator="in",
        value=["usgs-01017960"]
    )
]

In [ ]:
# Define the above percentile event detection calculated fields for 85th, 95th, and 99th percentiles.
# Note: both the threshold and event detection are based on the primary_value field.  
# This may differ from the way it is done in the NWM Explorer.  Does the NWM Explorer use the primary_value 
# of the threshold definition but the secondary_value field for event detection?

remove_for_quantiles = ["secondary_location_id", "reference_time", "member"]
quantile_group = [c for c in uniquenes_fields if c not in remove_for_quantiles]

calculated_fields = [
    rcf.GenericSQL(
        output_field_name="quarter",
        sql_statement="CONCAT(YEAR(reference_time), '-Q', QUARTER(reference_time))"
    ),
    rcf.ForecastLeadTimeBins(
        bin_size=pd.Timedelta(hours=6),
        output_field_name="forecast_lead_time_bin"
    ),
    # tcf.AbovePercentileEventDetection(
    #     quantile=0.85,
    #     output_event_field_name="above_q85",
    #     skip_event_id=True,
    #     value_field_name="primary_value",
    #     uniqueness_fields=quantile_group
    # ),
    # tcf.AbovePercentileEventDetection(
    #     quantile=0.95,
    #     output_event_field_name="above_q95",
    #     skip_event_id=True,
    #     value_field_name="primary_value",
    #     uniqueness_fields=quantile_group
    # ),
    # tcf.AbovePercentileEventDetection(
    #     quantile=0.99,
    #     output_event_field_name="above_q99",
    #     skip_event_id=True,
    #     value_field_name="primary_value",
    #     uniqueness_fields=quantile_group
    # )
]

In [ ]:
# Get raw joined timeseries
tbl = ev.table("fcst_joined_timeseries").filter(filters).add_calculated_fields(calculated_fields, engine="python")

In [ ]:
tbl.select("reference_time", "value_time", "forecast_lead_time_bin").order_by(["reference_time", "value_time"]).show()

In [ ]:
# len(tbl.distinct_values("primary_location_id"))

In [ ]:
# Stack thresholds
threshold_cols = ["above_q85", "above_q95", "above_q99"]
threshold_stack_base_cols = [c for c in tbl.columns if c not in threshold_cols]
# threshold_stack_base_cols

In [ ]:
joined_timeseries_with_thresholds_tbl = (
    tbl.selectExpr(
        *threshold_stack_base_cols,
        """
        stack(
            4,
            cast(null as string), true,
            'above_q85', above_q85,
            'above_q95', above_q95,
            'above_q99', above_q99
        ) as (threshold, keep_row)
        """
    )
    .where("keep_row")
    .select(*threshold_stack_base_cols, "threshold")   # no .drop()
)

# print(f"no threshold_rows: {joined_timeseries_with_thresholds_tbl.where("threshold is NULL").count()}")
# print(f"threshold_rows: {joined_timeseries_with_thresholds_tbl.where("threshold is not NULL").count()}")
# print(f"total: {joined_timeseries_with_thresholds_tbl.count()}")


In [ ]:
# Add window aggregations
window_metrics = [
    s.Average(
        primary_field_name="primary_value",
        output_field_name="mean_primary_value"
    ),
    s.Average(
        primary_field_name="secondary_value",
        output_field_name="mean_secondary_value"
    ),
    s.Minimum(
        primary_field_name="primary_value",
        output_field_name="min_primary_value"
    ),
    s.Minimum(
        primary_field_name="secondary_value",
        output_field_name="min_secondary_value"
    ),
    s.Maximum(
        primary_field_name="primary_value",
        output_field_name="max_primary_value"
    ),
    s.Maximum(
        primary_field_name="secondary_value",
        output_field_name="max_secondary_value"
    ),
    s.Count(
        primary_field_name="secondary_value",
        output_field_name="n_in_bin"
    )
]

In [ ]:
group_by_bin = [*uniquenes_fields, "quarter", "forecast_lead_time_bin", "threshold"]
group_by_bin

In [ ]:
bin_aggs_tbl = joined_timeseries_with_thresholds_tbl.aggregate(
    group_by=group_by_bin,
    metrics=window_metrics
)
# print(f"bin_aggs: {bin_aggs_tbl.count()}")

In [ ]:
# after your bin aggregation
# bin_aggs_tbl.select("n_in_bin").summary().show()

In [ ]:
pivoted_bin_aggs_tbl = bin_aggs_tbl.selectExpr(
    *group_by_bin,
    """
    stack(
        3,
        'mean', mean_primary_value, mean_secondary_value,
        'min',  min_primary_value,  min_secondary_value,
        'max',  max_primary_value,  max_secondary_value
    ) as (window_agg, primary_value, secondary_value)
    """
)
# print(f"pivoted_bin_aggs: {pivoted_bin_aggs_tbl.count()}")
# pivoted_bin_aggs_tbl.columns

In [ ]:
# pivoted_bin_aggs_tbl.show()

In [ ]:
# pivoted_bin_aggs_tbl.distinct_values("primary_location_id")
# pivoted_bin_aggs_tbl.distinct_values("reference_time")

In [ ]:
# Nash-Sutcliffe efficiency
# Relative mean bias
# Pearson correlation coefficient
# Kling-Gupta efficiency

# Relative mean
# Relative median
# Relative minimum
# Relative maximum
# Relative standard deviation


# Configure bootstrap
bootstap = bs.Stationary(
    reps=1000,
    # block_size=100,
    seed=1234,
    quantiles=[0.025, 0.975]
)

In [ ]:
metrics = [
    s.Count(),
    s.Average(),
    s.Minimum(),
    s.Maximum(),
    dm.RelativeMean(),
    dm.RelativeMedian(),
    dm.RelativeMinimum(),
    dm.RelativeMaximum(),
    dm.RelativeStandardDeviation(),
    dm.RelativeBias(
        add_epsilon=True,
    ),
    dm.NashSutcliffeEfficiency(
        add_epsilon=True,
    ),
    dm.KlingGuptaEfficiency(
        add_epsilon=True,
    ),
    dm.PearsonCorrelation(
        add_epsilon=True,
    ),
    # NOTE: unpack_results is intentionally NOT set on the bootstrap metrics below.
    # teehr's default unpack path (post_process_metric_results -> unpack_sdf_dict_columns)
    # calls sdf.select(column_name).first() once per metric with unpack_results=True -- a
    # real Spark action that retriggers the entire upstream lazy DAG once per metric (9x
    # here) and is the confirmed cause of the "ShuffleMapStage ... first at
    # teehr/querying/utils.py:207" crashes and nondeterministic same-config failures seen
    # in the profiling table above. We unpack manually after aggregation instead (see the
    # unpack_quantile_bootstrap_columns cell below), which needs no Spark action since the
    # quantile keys are already known statically from `bootstap.quantiles`.
    dm.RelativeMean(
        output_field_name="relative_mean_boot",
        bootstrap=bootstap,
    ),
    dm.RelativeMedian(
        output_field_name="relative_median_boot",
        bootstrap=bootstap,
    ),
    dm.RelativeMinimum(
        output_field_name="relative_minimum_boot",
        bootstrap=bootstap,
    ),
    dm.RelativeMaximum(
        output_field_name="relative_maximum_boot",
        bootstrap=bootstap,
    ),
    dm.RelativeStandardDeviation(
        output_field_name="relative_standard_deviation_boot",
        bootstrap=bootstap,
    ),
    dm.NashSutcliffeEfficiency(
        output_field_name="nash_sutcliffe_efficiency_boot",
        bootstrap=bootstap,
    ),
    dm.RelativeBias(
        output_field_name="relative_bias_boot",
        bootstrap=bootstap,
    ),
    dm.PearsonCorrelation(
        output_field_name="pearson_correlation_boot",
        bootstrap=bootstap,
    ),
    dm.KlingGuptaEfficiency(
        output_field_name="kling_gupta_efficiency_boot",
        bootstrap=bootstap,
    ),
]

In [ ]:
group_by = [
    "primary_location_id",
    "secondary_location_id",
    "configuration_name",
    "unit_name",
    "variable_name",
    "member",
    "quarter",
    "forecast_lead_time_bin",
    "threshold",
    "window_agg",
]

In [ ]:
%%time
results = pivoted_bin_aggs_tbl.aggregate(
    group_by=group_by,
    metrics=metrics
)

In [ ]:
# Manually unpack the bootstrap quantile MapType columns instead of relying on
# `unpack_results=True` (see note above the `metrics` list for why: the default
# unpack path triggers a Spark .first() action per metric that retriggers the
# whole upstream DAG). Quantile keys are known statically from `bootstap.quantiles`,
# so no action is needed here -- this stays fully lazy.
#
# TODO: teehr now derives these keys statically inside `aggregate()` itself, so
# once `devTeehrVersion` in project.garden.yml is bumped past that fix and the
# Jupyter/Spark images are rebuilt, delete this cell and set
# `unpack_results=True` on the bootstrap metrics instead.
#
# NOTE: `results` is a teehr BaseTable, not a plain PySpark DataFrame. BaseTable.drop()
# is a different method (drops the underlying warehouse table, no column arg) that
# shadows PySpark's DataFrame.drop(*cols) even with enable_spark_proxy=True, since the
# proxy in __getattr__ only kicks in when the attribute isn't already defined on the
# class. So we do the column manipulation on the raw sdf via .to_sdf() and rewrap once
# with ._with_sdf() at the end, which is the same internal pattern order_by/aggregate/
# add_geometry already use.
def unpack_quantile_bootstrap_columns(table, metrics):
    sdf = table.to_sdf()
    for m in metrics:
        if not getattr(m, "bootstrap", None):
            continue
        for q in m.bootstrap.quantiles:
            key = f"{m.output_field_name}_{q}"
            sdf = sdf.withColumn(key.replace(".", "_"), F.col(m.output_field_name).getItem(key))
        sdf = sdf.drop(m.output_field_name)
    return table._with_sdf(sdf)

results = unpack_quantile_bootstrap_columns(results, metrics)

In [ ]:
results = results.order_by(group_by).add_geometry()

In [ ]:
%%time
results.explain(mode="simple")

In [ ]:
# %%time
# results.show()

In [ ]:
# NOTE: pointed at a *_test table while validating the unpack_results/.first() fix so we
# don't overwrite the useful existing results in nwmd_metrics_by_location. Repoint back to
# "nwmd_metrics_by_location" only after full-scale validation succeeds consistently.
table_name = "nwmd_metrics_by_location_test"

nullables = ["member", "threshold"]
table_exists = ev.spark.catalog.tableExists(f"iceberg.teehr.{table_name}")

if table_exists:
    results.write_to(
        table_name=table_name,
        write_mode="upsert",
        uniqueness_fields=[column for column in group_by if column not in nullables],
        nullable_fields=nullables,
        partition_by=["quarter"],
    )
else:
    results.write_to(
        table_name=table_name,
        write_mode="create_or_replace",
        partition_by=["quarter"],
    )

# Read the metric column names off the written result rather than off the metric
# models. The bootstrap metrics' MapType columns are replaced by one column per
# quantile (e.g. relative_mean_boot -> relative_mean_boot_0_025, _0_975), so
# `metric.output_field_name` would advertise columns that don't exist in the
# table. `.columns` is schema-only, so this costs no Spark action.
# "name" and "geometry" come from add_geometry(), not from a metric.
non_metric_columns = set(group_by) | {"name", "geometry"}
metric_columns = [
    c for c in results.to_sdf().columns if c not in non_metric_columns
]

properties = {
    "description": "NWM diagnostics metrics by location ID",
    "group_by": ", ".join(group_by),
    "metrics": ", ".join(metric_columns)
}

for key, value in properties.items():
    ev.spark.sql(f"""
        ALTER TABLE iceberg.teehr.{table_name} SET TBLPROPERTIES ('{key}' = '{value}')
    """)


In [ ]:
end = time.perf_counter()

elapsed_seconds = end - start
print(f"{elapsed_seconds:.6f} s")

In [ ]:
# Capture resource-usage metrics for this run BEFORE spark.stop() (the REST API
# stops responding once the session ends). Paste the printed markdown row into
# the Profiling table above to keep a running record.
n_locations = len(location_ids) if "location_ids" in globals() else "All"
n_days = _infer_days_from_filters(filters)

run_metrics = capture_spark_run_metrics(spark, label=f"{n_locations} locations, {n_days} days")
report_utilization(run_metrics, elapsed_seconds)

print(
    f"\n| {n_locations} | {n_days} | {bootstap.reps} | Cluster - {spark_config_summary(spark)} "
    f"| {elapsed_seconds:.0f}s | (see run_metrics above) |"
)

In [ ]:
# Run this against the still-live session (spark.stop() is commented out below)
# to see the actual failure reason for the retried/failed stages from this run,
# without needing to read it off the Spark UI by hand.
stage_failures = get_stage_attempt_failures(spark)

In [ ]:
spark.stop()